In [0]:
# Read the validated bronze records (batch 1 good data) into a Spark DataFrame for silver-layer cleaning
silver_df = spark.read.table('it_support_tickets.bronze.bronze_table')

In [0]:
# Preview the bronze data before any silver cleaning is applied
display(silver_df)

In [0]:
# Standardize text formatting: trim whitespace on all text columns, uppercase ticket_id/employee_id/support_category/priority,
# uppercase ticket_status and replace any spaces with underscores, and title-case location
from pyspark.sql.functions import *

silver_df = silver_df.withColumn(
    "ticket_id", upper(trim(col("ticket_id")))
).withColumn(
    "employee_id", upper(trim(col("employee_id")))
).withColumn(
    "support_category", upper(trim(col("support_category")))
).withColumn(
    "priority", upper(trim(col("priority")))
).withColumn(
    "ticket_status",
    regexp_replace(upper(trim(col("ticket_status"))), " ", "_")
).withColumn(
    "location", initcap(trim(col("location")))
).withColumn(
    "support_engineer", trim(col("support_engineer"))
)

In [0]:
# Correct inconsistent category and priority values (HARDWRE -> HARDWARE, NETWORKING -> NETWORK, URGENT -> CRITICAL) so each group has one consistent label
silver_df = silver_df.withColumn(
    "support_category",
    when(col("support_category") == "HARDWRE", "HARDWARE")
    .when(col("support_category") == "NETWORKING", "NETWORK")
    .otherwise(col("support_category"))
)
silver_df = silver_df.withColumn(
    "priority",
    when(col("priority") == "URGENT", "CRITICAL")
    .otherwise(col("priority"))
)

In [0]:
# Correct remaining typos and status values (CRITCAL -> CRITICAL, 'IN PROGRESS' -> IN_PROGRESS, 'CLOSED??' -> RESOLVED)
silver_df = silver_df.withColumn(
    "priority",
    when(col("priority") == "CRITCAL", "CRITICAL")
    .otherwise(col("priority"))
).withColumn(
    "ticket_status",
    when(col("ticket_status") == "IN PROGRESS", "IN_PROGRESS" )
    .when(col("ticket_status") == "CLOSED??", "RESOLVED")
    .otherwise(col("ticket_status"))
)

In [0]:
# Preview the data after text standardization and value corrections
display(silver_df)

In [0]:
# Derive a new business column: total_resolution_cost = resolution_hours * hourly_cost
silver_df = silver_df.withColumn(
    "total_resolution_cost",
    col("resolution_hours") * col("hourly_cost")
)

In [0]:
# Preview the data with the new derived cost column
display(silver_df)

In [0]:
# Validate the final silver data: preview it, print the record count, and show the final schema
display(silver_df)

print("Silver record count:", silver_df.count())

silver_df.printSchema()

In [0]:
# Write the cleaned and enriched data to the silver Delta table, overwriting it on each run
silver_df.write.mode("overwrite").format('delta').option('mergeSchema','true').saveAsTable("it_support_tickets.silver.silver_table")

In [0]:
%sql
-- Preview the saved silver table with a quick SQL query
select * from it_support_tickets.silver.silver_table